**Вариант 10**

**Практическая работа 4. Временные ряды и пользовательские сессии**

Цель работы: освоить работу с датами и временем в pandas, агрегирование по временным интервалам и анализ связи между активностью пользователей (событиями) и продажами (выручкой).

Задание варианта 10:

1. объединить дневные агрегаты заказов и событий;
2. посчитать корреляцию между количеством событий и суммой выручки по дням;
3. интерпретировать результат.

Исходные данные

Используются файлы из папки `for_lab_3` — те же наборы, что и в практической работе 3:

| Файл | Формат | Содержимое |
|---|---|---|
| `orders_2024_01.csv`, `orders_2024_02.csv`, `orders_2024_03.csv` | CSV | заказы интернет-магазина по месяцам |
| `events_records.json` | JSON (records) | события/клики пользователей |
| `events_lines.json` | JSON Lines | события/клики пользователей |

Структура заказов: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status`.

Структура событий: `event_id`, `user_id`, `event_time`, `event_type`, `page`.

В заказах 15 000 позиций за январь–март 2024 года, в событиях — 4 000 записей.

In [1]:
import glob
import json
import os

import numpy as np
import pandas as pd

# папка с исходными файлами практической работы
DATA_DIR = 'for_lab_3'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

print('Рабочая папка с данными:', os.path.abspath(DATA_DIR))
print()
for path in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    print(f'{os.path.basename(path):<25}{os.path.getsize(path):>10} байт')

Рабочая папка с данными: D:\учёба\vsuet\ims\for_lab_3

events_lines.json            239555 байт
events_records.json          224211 байт
orders_2024_01.csv           244027 байт
orders_2024_02.csv           245248 байт
orders_2024_03.csv           250278 байт
products.xlsx                  6526 байт
user_profile.parquet          11919 байт


**1. Объединить дневные агрегаты заказов и событий**

In [2]:
orders_files = sorted(glob.glob(os.path.join(DATA_DIR, 'orders_2024_*.csv')))

orders_parts = []
for path in orders_files:
    part = pd.read_csv(path, parse_dates=['order_datetime'])
    orders_parts.append(part)
    print(f'[+] {os.path.basename(path)}: {part.shape[0]} строк, {part.shape[1]} столбцов')

# объединяем все месяцы в одну таблицу
df_orders = pd.concat(orders_parts, ignore_index=True)

# выручка по каждой позиции заказа
df_orders['revenue'] = df_orders['price'] * df_orders['quantity']

print()
print('Заказы:', df_orders.shape)
print('Период:', df_orders['order_datetime'].min(), '—', df_orders['order_datetime'].max())
print('Уникальных заказов:', df_orders['order_id'].nunique())
print()
print('Статусы заказов:')
print(df_orders['status'].value_counts().to_string())
print()
print(df_orders.head().to_string())

[+] orders_2024_01.csv: 5000 строк, 7 столбцов
[+] orders_2024_02.csv: 5000 строк, 7 столбцов
[+] orders_2024_03.csv: 5000 строк, 7 столбцов

Заказы: (15000, 8)
Период: 2024-01-01 00:10:25 — 2024-03-31 23:34:00
Уникальных заказов: 15000

Статусы заказов:
status
paid         12032
cancelled     2212
refunded       756

   order_id  user_id      order_datetime  product_id     price  quantity    status   revenue
0         1     1044 2024-01-03 18:39:26          26  14038.15         1  refunded  14038.15
1         2     1079 2024-01-03 17:45:43          16   9172.60         1      paid   9172.60
2         3     1041 2024-01-06 22:46:38          25   9035.26         2      paid  18070.52
3         4     1083 2024-01-08 21:03:51           9  15234.33         3      paid  45702.99
4         5     1068 2024-01-03 23:26:53          24   4134.38         2      paid   8268.76


In [3]:
def read_events_records(path):
    '''Читает JSON в формате records (список объектов).'''
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)

    df = pd.DataFrame(raw)

    # файл может быть сохранён транспонированным: 5 строк-признаков и 2000 столбцов
    if 'event_id' not in df.columns:
        df = df.T.reset_index(drop=True)
        df.columns = ['event_id', 'user_id', 'event_time', 'event_type', 'page']

    return df


# --- формат records ---
events_records = read_events_records(os.path.join(DATA_DIR, 'events_records.json'))
events_records['event_id'] = events_records['event_id'].astype('int64')
events_records['user_id'] = events_records['user_id'].astype('int32')

# --- формат JSON Lines ---
events_lines = pd.read_json(os.path.join(DATA_DIR, 'events_lines.json'), lines=True)
events_lines['event_id'] = events_lines['event_id'].astype('int64')
events_lines['user_id'] = events_lines['user_id'].astype('int32')

print('events_records.json:', events_records.shape)
print(events_records.head(3).to_string())
print()
print('events_lines.json:  ', events_lines.shape)
print(events_lines.head(3).to_string())

# объединяем оба источника событий
df_events = pd.concat([events_records, events_lines], ignore_index=True)
df_events['event_time'] = pd.to_datetime(df_events['event_time'])

print()
print('Всего событий:  ', len(df_events))
print('Период событий: ', df_events['event_time'].min(), '—', df_events['event_time'].max())
print('Типы событий:')
print(df_events['event_type'].value_counts().to_string())

events_records.json: (2000, 5)
   event_id  user_id           event_time   event_type         page
0         1     1085  2024-01-30T20:37:32    page_view  /product/26
1         2     1006  2024-02-24T22:55:13  add_to_cart  /product/28
2         3     1024  2024-02-14T17:49:43        click  /product/27

events_lines.json:   (2000, 5)
   event_id  user_id          event_time   event_type         page
0      2001     1029 2024-03-19 00:34:38     purchase  /product/20
1      2002     1056 2024-03-25 17:16:08  add_to_cart        /cart
2      2003     1081 2024-02-06 06:09:49  add_to_cart  /product/30

Всего событий:   4000
Период событий:  2024-01-01 01:06:07 — 2024-03-31 23:39:39
Типы событий:
event_type
click          819
purchase       807
page_view      804
scroll         788
add_to_cart    782


In [4]:
# на всякий случай приводим время к datetime (CSV уже разобран через parse_dates)
df_orders['order_datetime'] = pd.to_datetime(df_orders['order_datetime'])
df_events['event_time'] = pd.to_datetime(df_events['event_time'])

# локализуем как UTC и переводим в московское время
df_orders['order_datetime_utc'] = df_orders['order_datetime'].dt.tz_localize('UTC')
df_orders['order_datetime_msk'] = df_orders['order_datetime_utc'].dt.tz_convert('Europe/Moscow')

df_events['event_time_utc'] = df_events['event_time'].dt.tz_localize('UTC')
df_events['event_time_msk'] = df_events['event_time_utc'].dt.tz_convert('Europe/Moscow')

# дата, час и день недели по московскому времени
df_orders['order_date'] = df_orders['order_datetime_msk'].dt.date
df_orders['order_hour'] = df_orders['order_datetime_msk'].dt.hour
df_orders['order_weekday'] = df_orders['order_datetime_msk'].dt.day_name()

df_events['event_date'] = df_events['event_time_msk'].dt.date
df_events['event_hour'] = df_events['event_time_msk'].dt.hour
df_events['event_weekday'] = df_events['event_time_msk'].dt.day_name()

print('Заказы:')
print(df_orders[['order_datetime', 'order_datetime_msk', 'order_date', 'order_hour', 'order_weekday']].head().to_string())
print()
print('События:')
print(df_events[['event_time', 'event_time_msk', 'event_date', 'event_hour', 'event_weekday']].head().to_string())
print()
print('Часовой пояс заказов: ', df_orders['order_datetime_msk'].dt.tz)
print('Часовой пояс событий:', df_events['event_time_msk'].dt.tz)

Заказы:
       order_datetime        order_datetime_msk  order_date  order_hour order_weekday
0 2024-01-03 18:39:26 2024-01-03 21:39:26+03:00  2024-01-03          21     Wednesday
1 2024-01-03 17:45:43 2024-01-03 20:45:43+03:00  2024-01-03          20     Wednesday
2 2024-01-06 22:46:38 2024-01-07 01:46:38+03:00  2024-01-07           1        Sunday
3 2024-01-08 21:03:51 2024-01-09 00:03:51+03:00  2024-01-09           0       Tuesday
4 2024-01-03 23:26:53 2024-01-04 02:26:53+03:00  2024-01-04           2      Thursday

События:
           event_time            event_time_msk  event_date  event_hour event_weekday
0 2024-01-30 20:37:32 2024-01-30 23:37:32+03:00  2024-01-30          23       Tuesday
1 2024-02-24 22:55:13 2024-02-25 01:55:13+03:00  2024-02-25           1        Sunday
2 2024-02-14 17:49:43 2024-02-14 20:49:43+03:00  2024-02-14          20     Wednesday
3 2024-02-16 11:21:32 2024-02-16 14:21:32+03:00  2024-02-16          14        Friday
4 2024-02-08 05:25:21 2024-02-08 08:

In [5]:
# выручка только по оплаченным заказам — понадобится для проверки устойчивости результата
df_orders['paid_revenue'] = df_orders['revenue'].where(df_orders['status'] == 'paid', 0)

# временной индекс для ресемплинга
orders_ts = df_orders.set_index('order_datetime_msk').sort_index()
events_ts = df_events.set_index('event_time_msk').sort_index()

# дневные агрегаты заказов
daily_orders = (
    orders_ts
    .resample('D')
    .agg(
        orders_count=('order_id', 'nunique'),
        revenue_sum=('revenue', 'sum'),
        paid_revenue_sum=('paid_revenue', 'sum'),
    )
    .reset_index()
    .rename(columns={'order_datetime_msk': 'date'})
)

# дневные агрегаты событий
daily_events = (
    events_ts
    .resample('D')
    .agg(events_count=('event_id', 'count'))
    .reset_index()
    .rename(columns={'event_time_msk': 'date'})
)

# объединяем агрегаты по дате
daily = (
    pd.merge(daily_orders, daily_events, on='date', how='outer')
    .sort_values('date')
    .reset_index(drop=True)
)

# дни без заказов/событий получают 0
daily[['orders_count', 'events_count']] = (
    daily[['orders_count', 'events_count']].fillna(0).astype(int)
)
daily[['revenue_sum', 'paid_revenue_sum']] = daily[['revenue_sum', 'paid_revenue_sum']].fillna(0)

daily_view = daily.copy()
daily_view['date'] = daily_view['date'].dt.strftime('%Y-%m-%d')

print('Дневная таблица:', daily.shape)
print('Период:', daily['date'].min().date(), '—', daily['date'].max().date())
print()
print('Первые 10 дней:')
print(daily_view.head(10).to_string(index=False))
print()
print('Последние 5 дней:')
print(daily_view.tail(5).to_string(index=False))
print()
print('Описательная статистика:')
print(daily[['orders_count', 'events_count', 'revenue_sum']].describe().round(2).to_string())

Дневная таблица: (92, 5)
Период: 2024-01-01 — 2024-04-01

Первые 10 дней:
      date  orders_count  revenue_sum  paid_revenue_sum  events_count
2024-01-01           141   3888229.53        3223253.97            26
2024-01-02           165   4242645.43        3628756.60            29
2024-01-03           169   4517004.59        3461925.31            44
2024-01-04           168   3624982.14        2867880.15            35
2024-01-05           181   4763831.59        3857466.11            31
2024-01-06           160   4450229.73        3422647.21            39
2024-01-07           158   3967384.29        3131348.25            27
2024-01-08           161   4201864.84        3015189.58            35
2024-01-09           175   4442862.12        3476871.43            27
2024-01-10           169   4249550.74        3237252.17            28

Последние 5 дней:
      date  orders_count  revenue_sum  paid_revenue_sum  events_count
2024-03-28           180   4706193.65        3509836.18            

**2. Посчитать корреляцию между количеством событий и суммой выручки по дням**

In [6]:
pearson_r = daily['events_count'].corr(daily['revenue_sum'])
spearman_r = daily['events_count'].rank().corr(daily['revenue_sum'].rank())
pearson_paid = daily['events_count'].corr(daily['paid_revenue_sum'])
corr_events_orders = daily['events_count'].corr(daily['orders_count'])
corr_orders_revenue = daily['orders_count'].corr(daily['revenue_sum'])

print(f'Корреляция по дням (n = {len(daily)}):')
print(f'  Пирсон: события — выручка:            {pearson_r: .3f}')
print(f'  Спирмен: события — выручка:           {spearman_r: .3f}')
print(f'  Пирсон: события — оплаченная выручка: {pearson_paid: .3f}')
print(f'  Пирсон: события — число заказов:      {corr_events_orders: .3f}')
print(f'  Пирсон: число заказов — выручка:      {corr_orders_revenue: .3f}')

# проверка без неполных первого и последнего дней
complete_days = daily.iloc[1:-1]
pearson_complete = complete_days['events_count'].corr(complete_days['revenue_sum'])
print()
print(f'Пирсон без неполных первого и последнего дней (n = {len(complete_days)}): {pearson_complete: .3f}')

print()
print('Корреляционная матрица (Пирсон):')
print(daily[['events_count', 'orders_count', 'revenue_sum']].corr().round(3).to_string())

Корреляция по дням (n = 92):
  Пирсон: события — выручка:             0.293
  Спирмен: события — выручка:            0.204
  Пирсон: события — оплаченная выручка:  0.268
  Пирсон: события — число заказов:       0.407
  Пирсон: число заказов — выручка:       0.882

Пирсон без неполных первого и последнего дней (n = 90):  0.197

Корреляционная матрица (Пирсон):
              events_count  orders_count  revenue_sum
events_count         1.000         0.407        0.293
orders_count         0.407         1.000        0.882
revenue_sum          0.293         0.882        1.000


**3. Интерпретировать результат**

In [7]:
print('Топ-5 дней по количеству событий:')
print(
    daily_view
    .nlargest(5, 'events_count')
    [['date', 'events_count', 'orders_count', 'revenue_sum']]
    .to_string(index=False)
)

print()
print('Топ-5 дней по выручке:')
print(
    daily_view
    .nlargest(5, 'revenue_sum')
    [['date', 'events_count', 'orders_count', 'revenue_sum']]
    .to_string(index=False)
)

# месячные итоги только по полным месяцам январь–март
monthly = (
    daily
    .set_index('date')[['orders_count', 'events_count', 'revenue_sum']]
    .resample('MS')
    .sum()
)
monthly = monthly[monthly.index.month.isin([1, 2, 3])]
monthly.index = monthly.index.strftime('%Y-%m')

print()
print('Итоги по месяцам:')
print(monthly.round(2).to_string())

print()
print(f'Средняя выручка на один заказ по дням: {(daily["revenue_sum"] / daily["orders_count"]).mean():,.2f} руб.')

Топ-5 дней по количеству событий:


      date  events_count  orders_count  revenue_sum
2024-02-09            87           174   4254331.41
2024-02-22            82           180   4707818.97
2024-02-12            75           178   5087744.24
2024-02-21            75           172   4190510.63
2024-02-04            74           148   3597430.17

Топ-5 дней по выручке:
      date  events_count  orders_count  revenue_sum
2024-02-27            61           184   5380840.41
2024-02-01            67           199   5345978.80
2024-01-14            48           181   5268443.08
2024-02-08            57           197   5150300.92
2024-02-12            75           178   5087744.24

Итоги по месяцам:
         orders_count  events_count   revenue_sum
date                                             
2024-01          4983          1024  1.289900e+08
2024-02          5001          1932  1.254424e+08
2024-03          4988          1037  1.293790e+08

Средняя выручка на один заказ по дням: 25,644.01 руб.
